# Phase 4a — Sample Extraction

**Runs locally** — requires access to `cocotext.v2.json`.

Produces two JSON files:
- `04_train_sample.json` — 300 train images with GT boxes and text
- `04_test_sample.json`  — 200 val images with GT boxes and text

Upload both to `outputs/results/` on Drive, then run `04b_full_benchmark.ipynb` on Colab.

## 1. Paths & Config

In [4]:
import json, random
from pathlib import Path
from collections import defaultdict

# ── Update these to match your local paths ──
COCO_DIR    = Path('../data/raw')
ANN_FILE    = COCO_DIR / 'cocotext.v2.json'
OUTPUT_DIR  = Path('../outputs/results')

SEED    = 42
N_TRAIN = 2000
N_TEST  = 1000

# Legibility bins — match full dataset distribution (from Phase 1 EDA)
# High  : >= 0.6 legible annotations
# Medium: 0.2 to 0.6
# Low   : < 0.2
BIN_THRESHOLDS = {'high': 0.6, 'medium': 0.2}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert ANN_FILE.exists(), f'Annotation file not found: {ANN_FILE}'
print('Paths OK ✓')


Paths OK ✓


## 2. Load & Filter Annotations

In [5]:
print('Loading COCO-Text annotations...')
with open(ANN_FILE) as f:
    coco = json.load(f)

# Build per-image annotation lookup
# Track both legible and all annotations to compute legibility ratio
img_all_anns     = defaultdict(int)   # total annotations per image
img_legible_anns = defaultdict(int)   # legible annotations per image
img_to_anns      = {}                 # legible+english+len>=2 annotations

for ann_id, ann in coco['anns'].items():
    iid = ann['image_id']
    img_all_anns[iid] += 1
    if ann.get('legibility') == 'legible':
        img_legible_anns[iid] += 1
    if ann.get('legibility') != 'legible': continue
    if ann.get('language')   != 'english': continue
    utf8 = ann.get('utf8_string', '').strip()
    if len(utf8) < 2: continue
    if iid not in img_to_anns:
        img_to_anns[iid] = []
    x, y, w, h = ann['bbox']
    img_to_anns[iid].append({
        'box':  [float(x), float(y), float(w), float(h)],
        'text': utf8
    })

# Compute legibility ratio per image
def legibility_bin(iid):
    total   = img_all_anns.get(iid, 0)
    legible = img_legible_anns.get(iid, 0)
    if total == 0: return 'low'
    ratio = legible / total
    if ratio >= BIN_THRESHOLDS['high']:   return 'high'
    if ratio >= BIN_THRESHOLDS['medium']: return 'medium'
    return 'low'

# Normalize keys
img_to_anns_int = {int(k): v for k, v in img_to_anns.items()}
imgs_int        = {int(k): v for k, v in coco['imgs'].items()}

# Split by set and bin
train_bins = defaultdict(list)
val_bins   = defaultdict(list)

for iid, info in imgs_int.items():
    if iid not in img_to_anns_int: continue
    b = legibility_bin(iid)
    if info.get('set') == 'train': train_bins[b].append(iid)
    elif info.get('set') == 'val': val_bins[b].append(iid)

for split, bins in [('Train', train_bins), ('Val', val_bins)]:
    total = sum(len(v) for v in bins.values())
    print(f'{split} eligible: {total} images')
    for b in ['high', 'medium', 'low']:
        print(f'  {b:6s}: {len(bins[b])} ({len(bins[b])/total*100:.1f}%)')


Loading COCO-Text annotations...
Train eligible: 12303 images
  high  : 5572 (45.3%)
  medium: 5489 (44.6%)
  low   : 1242 (10.1%)
Val eligible: 2899 images
  high  : 1311 (45.2%)
  medium: 1273 (43.9%)
  low   : 315 (10.9%)


## 3. Sample & Save JSONs

In [6]:
def stratified_sample(bins, n, seed=SEED):
    """
    Sample n images proportionally from high/medium/low bins.
    Preserves the legibility distribution of the full dataset.
    """
    random.seed(seed)
    total_eligible = sum(len(v) for v in bins.values())
    sampled = []
    remainder = {}

    for b in ['high', 'medium', 'low']:
        proportion = len(bins[b]) / total_eligible
        n_bin      = int(n * proportion)
        remainder[b] = (n * proportion) - n_bin
        n_bin = min(n_bin, len(bins[b]))
        sampled.extend(random.sample(bins[b], n_bin))

    # Fill remaining slots from bins with largest remainders
    still_needed = n - len(sampled)
    if still_needed > 0:
        for b in sorted(remainder, key=remainder.get, reverse=True):
            already = [s for s in sampled if s in bins[b]]
            pool    = [x for x in bins[b] if x not in already]
            take    = min(still_needed, len(pool))
            sampled.extend(random.sample(pool, take))
            still_needed -= take
            if still_needed == 0: break

    return sampled


train_sample = stratified_sample(train_bins, N_TRAIN)
test_sample  = stratified_sample(val_bins,   N_TEST)

# Report distribution
for label, sample, bins in [('Train', train_sample, train_bins),
                              ('Test',  test_sample,  val_bins)]:
    print(f'{label} sample: {len(sample)} images')
    for b in ['high', 'medium', 'low']:
        count = sum(1 for s in sample if s in bins[b])
        print(f'  {b:6s}: {count} ({count/len(sample)*100:.1f}%)')

def build_sample(ids):
    return [{
        'image_id':        iid,
        'file_name':       imgs_int[iid]['file_name'],
        'legibility_bin':  legibility_bin(iid),
        'gt':              img_to_anns_int[iid]
    } for iid in ids]

train_data = build_sample(train_sample)
test_data  = build_sample(test_sample)

train_out = OUTPUT_DIR / '04_train_sample.json'
test_out  = OUTPUT_DIR / '04_test_sample.json'

with open(train_out, 'w') as f: json.dump(train_data, f, indent=2)
with open(test_out,  'w') as f: json.dump(test_data,  f, indent=2)

print(f'\nTrain saved → {train_out}')
print(f'Test  saved → {test_out}')


Train sample: 2000 images
  high  : 905 (45.2%)
  medium: 892 (44.6%)
  low   : 203 (10.2%)
Test sample: 1000 images
  high  : 452 (45.2%)
  medium: 439 (43.9%)
  low   : 109 (10.9%)

Train saved → ..\outputs\results\04_train_sample.json
Test  saved → ..\outputs\results\04_test_sample.json


## 4. Copy Sampled Images for Upload

In [9]:
PROJECT_ROOT = Path('..')

In [11]:
import shutil

# ── Update to your local train2014 folder ──
IMG_SRC_DIR = PROJECT_ROOT / 'data/raw/train2014'
IMG_OUT_DIR = PROJECT_ROOT / 'outputs/benchmark_images'
# ───────────────────────────────────────────

IMG_OUT_DIR.mkdir(parents=True, exist_ok=True)

all_samples  = train_data + test_data
missing      = 0
copied       = 0

for s in all_samples:
    src = IMG_SRC_DIR / s['file_name']
    dst = IMG_OUT_DIR / s['file_name']
    if dst.exists():
        copied += 1
        continue
    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing += 1
        print(f'Missing: {src}')

print(f'\nCopied : {copied} images → {IMG_OUT_DIR}')
print(f'Missing: {missing}')
print('\n→ Upload outputs/benchmark_images/ to Drive at:')
print('  data/raw/benchmark_images/')
print('→ Upload outputs/results/04_train_sample.json and 04_test_sample.json to:')
print('  outputs/results/')


Copied : 3000 images → ..\outputs\benchmark_images
Missing: 0

→ Upload outputs/benchmark_images/ to Drive at:
  data/raw/benchmark_images/
→ Upload outputs/results/04_train_sample.json and 04_test_sample.json to:
  outputs/results/
